Analytical strategy
-------------------
The analysis tests three sequential claims:

  CLAIM 1 (Direct bias):
      Within each criminality category, CAP arrests are deported at higher
      rates than community enforcement arrests.
      Method: Cross-tabulation + two-sample t-tests.
      Effect size: ratio (CAP rate / Community rate) and absolute gap (pp).

  CLAIM 2 (Structural targeting):
      Nationality independently predicts pipeline assignment.
      Method: Cross-tabulation of pipeline share by citizenship country
              (top 15 by arrest volume).

  CLAIM 3 (Downstream consequence):
      Pipeline predicts enforcement program assignment (final_program).
      Method: Cross-tabulation of program distribution by pipeline.

Mediation (descriptive):
      Within-pipeline deportation rates by nationality estimate the
      pipeline-mediated share of the nationality-outcome gap.
      NOTE: This is descriptive mediation — no causal identification.
      The claim is about dataset structure, not causal effect.

No regression model is estimated in Phase 2. The analysis is deliberately
descriptive: the bias claim is about dataset structure, and regressing
outcome on pipeline would be circular (pipeline is part of the mechanism).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import numpy as np
import pandas as pd
from scipy import stats
from config import DATA_INTERIM, RESULTS_P2

CRIM_ORDER = [
    "1 Convicted Criminal",
    "2 Pending Criminal Charges",
    "3 Other Immigration Violator",
]

CRIM_SHORT = {
    "1 Convicted Criminal":        "Convicted Criminal",
    "2 Pending Criminal Charges":  "Pending Charges",
    "3 Other Immigration Violator":"Immigration Violator",
}


def load() -> pd.DataFrame:
    path = DATA_INTERIM / "arrests_classified.csv"
    if not path.exists():
        raise FileNotFoundError("Run 01_classify_pipelines.py first.")
    return pd.read_csv(path, low_memory=False)


def claim1_pipeline_deportation(df: pd.DataFrame) -> pd.DataFrame:


Within identical criminality labels, pipeline predicts deportation.


In [ ]:
    """Within identical criminality labels, pipeline predicts deportation."""
    print("CLAIM 1: Pipeline → Deportation (within criminality label)")
    print("="*60)

    rows = []
    ttest_rows = []

    for crim in CRIM_ORDER:
        sub = df[df["apprehension_criminality"] == crim]
        print(f"\n── {crim} (n={len(sub):,}) ──")

        for pipe in ["CAP", "Community", "287g", "Fugitive", "Border"]:
            s = sub[sub["pipeline"] == pipe]["deported"]
            if len(s) < 100:
                continue
            rows.append({
                "criminality":       crim,
                "pipeline":          pipe,
                "n":                 len(s),
                "deportation_rate":  round(s.mean(), 4),
                "std":               round(s.std(), 4),
            })

        # Key comparison: CAP vs Community
        cap = sub[sub["pipeline"] == "CAP"]["deported"]
        com = sub[sub["pipeline"] == "Community"]["deported"]
        if len(cap) > 0 and len(com) > 0:
            t_stat, p_val = stats.ttest_ind(cap, com, equal_var=False)
            ratio = cap.mean() / com.mean() if com.mean() > 0 else np.nan
            gap   = cap.mean() - com.mean()
            print(f"  CAP:       {cap.mean():.1%}  (n={len(cap):,})")
            print(f"  Community: {com.mean():.1%}  (n={len(com):,})")
            print(f"  Gap: {gap*100:+.1f}pp  Ratio: {ratio:.2f}×  t={t_stat:.1f}  p={p_val:.2e}")
            ttest_rows.append({
                "criminality":         crim,
                "cap_rate":            round(cap.mean(), 4),
                "community_rate":      round(com.mean(), 4),
                "absolute_gap_pp":     round(gap * 100, 2),
                "ratio":               round(ratio, 3),
                "cap_n":               len(cap),
                "community_n":         len(com),
                "t_statistic":         round(t_stat, 2),
                "p_value":             p_val,
                "p_value_sci":         f"{p_val:.2e}",
            })

    rates_df  = pd.DataFrame(rows)
    ttest_df  = pd.DataFrame(ttest_rows)
    rates_df.to_csv(RESULTS_P2 / "pipeline_deportation_rates.csv", index=False)
    ttest_df.to_csv(RESULTS_P2 / "ttest_results.csv", index=False)
    return rates_df


def claim2_nationality_pipeline(df: pd.DataFrame) -> pd.DataFrame:


Nationality determines pipeline assignment.


In [ ]:
    """Nationality determines pipeline assignment."""
    print("CLAIM 2: Nationality → Pipeline Assignment")
    print("="*60)

    top15 = df["citizenship_country"].value_counts().head(15).index
    sub15 = df[df["citizenship_country"].isin(top15)]

    nat_df = (
        sub15.groupby("citizenship_country")
        .agg(
            n=          ("pipeline", "count"),
            pct_cap=    ("pipeline", lambda x: round((x == "CAP").mean(), 4)),
            pct_community=("pipeline", lambda x: round((x == "Community").mean(), 4)),
            pct_287g=   ("pipeline", lambda x: round((x == "287g").mean(), 4)),
            pct_border= ("pipeline", lambda x: round((x == "Border").mean(), 4)),
            deportation_rate=("deported", lambda x: round(x.mean(), 4)),
        )
        .sort_values("pct_cap", ascending=False)
    )

    print(nat_df.to_string())
    nat_df.to_csv(RESULTS_P2 / "nationality_pipeline_deportation.csv")
    return nat_df


def claim3_program_assignment(df: pd.DataFrame) -> pd.DataFrame:


Pipeline determines enforcement program track.


In [ ]:
    """Pipeline determines enforcement program track."""
    print("CLAIM 3: Pipeline → Enforcement Program Assignment")
    print("="*60)

    prog_rows = []
    for pipe in ["CAP", "Community", "287g"]:
        sub = df[df["pipeline"] == pipe]
        prog_dist = sub["final_program"].value_counts(normalize=True)
        print(f"\n{pipe} (n={len(sub):,}):")
        print(prog_dist.head(5).apply(lambda x: f"{x:.1%}").to_string())
        for prog, share in prog_dist.items():
            prog_rows.append({"pipeline": pipe, "program": prog,
                               "share": round(share, 4), "n_pipe": len(sub)})

    prog_df = pd.DataFrame(prog_rows)
    prog_df.to_csv(RESULTS_P2 / "program_by_pipeline.csv", index=False)
    return prog_df


def descriptive_mediation(df: pd.DataFrame):


Within-pipeline deportation rates by nationality.

Shows what fraction of the nationality gap remains after controlling for pipeline - descriptive mediation only.
    


In [ ]:
    print("\n" + "="*60)
    print("DESCRIPTIVE MEDIATION: Pipeline-controlled nationality gaps")
    print("="*60)

    top5 = ["MEXICO", "HONDURAS", "GUATEMALA", "VENEZUELA", "PERU"]
    crim = "3 Other Immigration Violator"  # cleanest comparison (no criminal record)
    sub = df[(df["apprehension_criminality"] == crim) &
             (df["citizenship_country"].isin(top5))]

    print(f"\nWithin '{crim}', Community pipeline only:")
    com_sub = sub[sub["pipeline"] == "Community"]
    if len(com_sub) > 0:
        rates = com_sub.groupby("citizenship_country")["deported"].agg(["mean","count"])
        print(rates.sort_values("mean", ascending=False).to_string())

    print(f"\nWithin '{crim}', all pipelines (raw rates):")
    rates_all = sub.groupby("citizenship_country")["deported"].agg(["mean","count"])
    print(rates_all.sort_values("mean", ascending=False).to_string())


def main():
    df = load()
    print(f"Loaded {len(df):,} arrest records")
    print(f"Period: {df['year'].min():.0f}–{df['year'].max():.0f}")
    print(f"Pipelines: {df['pipeline'].value_counts().to_dict()}")

    claim1_pipeline_deportation(df)
    claim2_nationality_pipeline(df)
    claim3_program_assignment(df)
    descriptive_mediation(df)

    print(f"\nAll Phase 2 analysis results saved to {RESULTS_P2}")


if __name__ == "__main__":
    main()
